# 11. Full Chain Stress Test (Max Complexity)
This example demonstrates the maximum capability of `curaster` in a single chained pipeline. It reads a massive **XL** raster (16384 x 16384 pixels) directly from S3, applies a boolean spectral mask, clips it to a dynamic GeoJSON polygon, computes all **9** available terrain metrics (triggering multi-band GPU focal processing), and streams the result directly back to an S3 bucket.

## 1. Setup AWS Credentials and Input/Output URIs
Ensure your environment variables are configured. For `curaster` to write to S3, GDAL utilizes `/vsis3/` paths.

In [ ]:
import curaster
import os
import json

# Ensure AWS credentials are set (replace with your own if needed)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "")
os.environ.setdefault("AWS_DEFAULT_REGION",    "eu-central-1")

# Max test size (16384 x 16384) in the S3 benchmark bucket
input_s3 = "/vsis3/bucket/test_XL_16384x16384_lzw_b512_band.tif"

# Direct S3 write output path via GDAL VSI
output_s3 = "/vsis3/bucket/output/terrain_stress_XL_9band.tif"


## 2. Dynamic AOI Generation
We will extract the spatial extent of the input image without downloading the pixel data, using `get_info()`, and construct a polygon representing the inner 80% of the image.

In [ ]:
# Get info to dynamically create a clip polygon (reads only the GeoTIFF header)
info = curaster.open(input_s3).get_info()
gt = info["geotransform"]
w, h = info["width"], info["height"]

# Create an AOI covering the middle 80% of the image
x0 = gt[0] + gt[1] * (w * 0.1)
x1 = gt[0] + gt[1] * (w * 0.9)
y0 = gt[3] + gt[5] * (h * 0.1)
y1 = gt[3] + gt[5] * (h * 0.9)

aoi = json.dumps({
    "type": "Polygon",
    "coordinates": [[
        [x0, y0], [x1, y0], [x1, y1], [x0, y1], [x0, y0]
    ]]
})

print(f"Original size: {w} x {h}")


## 3. Build the Lazy Chain
The chain is not executed until `.save_s3()` is called. We will stack the following operations:
1. **`algebra()`**: Apply a conditional threshold mask (zeroing pixels below 500, preserving Band 1 otherwise).
2. **`clip()`**: Mask out all pixels outside the GeoJSON polygon we generated.
3. **`terrain()`**: Compute all 9 available terrain derivatives (`slope`, `aspect`, `hillshade`, `tri`, `tpi`, `roughness`, `prof_curv`, `plan_curv`, `total_curv`) using the Horn (3x3) method.

In [ ]:
chain = curaster.open(input_s3) \
    .algebra("(B1 > 500) * B1") \
    .clip(aoi) \
    .terrain(
        metrics=["slope", "aspect"],
        unit="degrees",
        method="horn"
    )

print("Lazy execution chain built successfully.")


## 4. Execute and Stream directly to S3
The operation will execute in chunks, utilizing multi-threaded parallel execution across the GPU and CPU to compute and upload the resulting **9-band** raster directly to S3 without overflowing system RAM. Total pixel throughput generated: **~2.4 Billion pixels!**

In [ ]:
%%time

# Execute the pipeline and stream the result directly back to S3
chain.save_s3(output_s3, verbose=True)

print(f"\nSuccessfully uploaded 9-band multi-metric terrain output to {output_s3}")


## 5. Visualizing a Chunk (Stream Inspection)
If you want to inspect what the pipeline is doing without writing the entire 16k x 16k image to disk, you can use `.iter_begin()` to stream the data, grab the very first processed chunk, and visualize it using `matplotlib`.

In [ ]:
import matplotlib.pyplot as plt

print("Starting iterator...")
# Start the iterator instead of save_s3
queue = chain.iter_begin(buf_chunks=2)

# Grab the very first chunk
first_chunk = queue.next()

if first_chunk:
    data = first_chunk["data"]
    y0 = first_chunk["y_offset"]
    chunk_h = first_chunk["height"]
    chunk_w = first_chunk["width"]
    
    print(f"Intercepted chunk! Shape: {data.shape} (Bands, Height, Width)")
    print(f"This chunk represents rows {y0} to {y0 + chunk_h} of the 16384x16384 output.")
    
    # Plot the first 3 terrain bands (Slope, Aspect, Hillshade)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Crop the chunk to a reasonable width for plotting
    plot_w = min(1500, chunk_w)
    
    axes[0].imshow(data[0, :500, :plot_w], cmap="plasma"); axes[0].set_title("Band 0: Slope")
    axes[1].imshow(data[1, :500, :plot_w], cmap="twilight"); axes[1].set_title("Band 1: Aspect")
    axes[2].imshow(data[2, :500, :plot_w], cmap="gray"); axes[2].set_title("Band 2: Hillshade")
    
    plt.tight_layout()
    plt.show()

# We can abandon the rest of the queue if we just wanted a preview!
